# Crucible Python Client Tutorial

This notebook demonstrates how to use the Crucible Python client to manage datasets, samples, projects, and their relationships.

**Prerequisites:**
- Crucible API credentials configured (run `crucible config init` in terminal)
- Access to project `crucible-demo`
- Example data files in the `data/` directory

## Table of Contents
1. [Setup and Configuration](#setup)
2. [Creating a Sample](#create-sample)
3. [Creating a Dataset](#create-dataset)
4. [Listing Datasets in a Project](#list-datasets)
5. [Getting a Dataset with Metadata](#get-dataset)
6. [Updating Dataset Metadata](#update-metadata)
7. [Downloading Dataset Files](#download-dataset)
8. [Linking a Sample to a Dataset](#link-sample-dataset)
9. [Linking Two Datasets (Parent-Child)](#link-datasets)
10. [Linking Two Samples (Parent-Child)](#link-samples)
11. [Adding a Thumbnail to a Dataset](#add-thumbnail)

<a id='setup'></a>
## 1. Setup and Configuration

First, you need to configure your Crucible API credentials. Run this command in your terminal (only needed once):

```bash
crucible config init
```

This will prompt you for:
- **API Key** - Get it from https://crucible.lbl.gov/api/v1/user_apikey
- **API URL** - Defaults to https://crucible.lbl.gov/api/v1
- **Default Project** (optional) - You can set `crucible-demo` as default

**Alternative: Without Terminal Access**

If you don't have terminal access (e.g., JupyterHub, Google Colab, VSCode Flatpak), you can initialize the client directly:

```python
client = CrucibleClient(
    api_url="https://crucible.lbl.gov/api/v1",
    api_key="your-api-key-here"
)
```

Once configured, you can import and use the Crucible client:

In [1]:
import os
from pathlib import Path
from crucible.client import CrucibleClient

# Initialize the client (automatically loads configuration)
client = CrucibleClient()

# Get the data directory path
EXAMPLES_DIR = Path(os.getcwd())
DATA_DIR = EXAMPLES_DIR / "data"

print("✓ Crucible client initialized successfully!")
print(f"Data directory: {DATA_DIR}")

✓ Crucible client initialized successfully!
Data directory: /home/roncofaber/software/nano-crucible/examples/data


Set your project ID for this tutorial:

In [2]:
# Project ID for this tutorial
PROJECT_ID = "crucible-demo"

# Verify project exists
project = client.projects.get(PROJECT_ID)
if project:
    print(f"✓ Working with project: {project['project_id']}")
    print(f"  Organization: {project.get('organization', 'N/A')}")
    print(f"  Lead: {project.get('project_lead_email', 'N/A')}")
else:
    print(f"⚠ Project '{PROJECT_ID}' not found. You may need to create it first.")

✓ Working with project: crucible-demo
  Organization: Molecular Foundry
  Lead: roncoroni@lbl.gov


<a id='create-sample'></a>
## 2. Creating a Sample

Samples represent physical materials or specimens. Let's create a sample:

In [ ]:
# Create a sample
sample = client.samples.create(
    sample_name="Silicon Wafer A - Tutorial Example",
    project_id=PROJECT_ID,
    description="Silicon wafer for thermal conductivity measurements (tutorial example)",
    timestamp="2024-01-15",  # when the sample was fabricated/collected
)

sample_id = sample['unique_id']
print(f"✓ Sample created successfully!")
print(f"  Sample ID: {sample_id}")
print(f"  Sample Name: {sample['sample_name']}")
print(f"  Project: {sample['project_id']}")
print(f"  Timestamp: {sample.get('timestamp', 'N/A')}")

<a id='create-dataset'></a>
## 3. Creating a Dataset

Datasets contain data files and metadata. You can create a dataset with or without files.

### 3.1 Create Dataset with Metadata Only

In [ ]:
from crucible.models import Dataset

# Define dataset metadata
dataset_metadata = Dataset(
    project_id=PROJECT_ID,
    measurement="thermal_conductivity",
    dataset_name="Thermal Conductivity Measurement - Sample A (Tutorial)",
    timestamp="2024-01-15T10:30:00",  # when the measurement was taken
    public=False
)

# Create dataset without files
result = client.datasets.create(
    dataset=dataset_metadata,
    scientific_metadata={
        "temperature_range": "273-363 K",
        "measurement_method": "3-omega method",
        "equipment": "Lakeshore 336 + SR830 lock-in",
        "sample_type": "silicon wafer"
    },
    keywords=["thermal", "conductivity", "silicon", "tutorial"]
)

dataset_id = result['dsid']
print(f"✓ Dataset created successfully!")
print(f"  Dataset ID: {dataset_id}")
print(f"  Dataset Name: {result['created_record']['dataset_name']}")
print(f"  Timestamp: {result['created_record'].get('timestamp', 'N/A')}")

### 3.2 Create Dataset with Files Using a Parser

Now let's create a dataset with actual data files using the `BaseParser`. Parsers handle file upload and ingestion, and can optionally extract metadata from domain-specific formats. The `BaseParser` is a generic pass-through that uploads files without any format-specific parsing — use it when you just want to upload files as-is.

For domain-specific formats (e.g. LAMMPS), swap `BaseParser` with the corresponding parser class (e.g. `LAMMPSParser`).

In [ ]:
from crucible.parsers import BaseParser

# Create a parser for the files
# BaseParser = generic upload with no format-specific parsing
# For domain-specific formats, replace BaseParser with e.g. LAMMPSParser
parser = BaseParser(
    files_to_upload=[
        str(DATA_DIR / "thermal_conductivity_data.csv"),
        str(DATA_DIR / "measurement_notes.txt")
    ],
    project_id=PROJECT_ID,
    measurement="thermal_conductivity",
    dataset_name="Thermal Conductivity Data with Files (Tutorial)",
    timestamp="2024-01-15T14:00:00",  # when the measurement was taken
    # If timestamp is omitted, it is auto-set from the main file's modification time
    public=False,
    metadata={
        "temperature_range": "273-363 K",
        "data_points": 10,
        "measurement_method": "3-omega method",
        "sample_material": "silicon"
    },
    keywords=["thermal", "conductivity", "data", "tutorial"]
)

print("Files to upload:")
for f in parser.files_to_upload:
    exists = "✓" if Path(f).exists() else "✗"
    print(f"  {exists} {Path(f).name}")

# Upload the parsed dataset to Crucible
result_with_files = parser.upload_dataset(wait_for_ingestion_response=True)

dataset_with_files_id = result_with_files['dsid']
print(f"\n✓ Dataset with files created successfully!")
print(f"  Dataset ID: {dataset_with_files_id}")
print(f"  Files uploaded: {len(result_with_files.get('uploaded_files', []))}")
print(f"  Ingestion status: {result_with_files.get('ingestion_request', {}).get('status', 'N/A')}")

<a id='list-datasets'></a>
## 4. Listing Datasets in a Project

Retrieve all datasets associated with a project:

In [6]:
# List all datasets in the project
datasets = client.datasets.list(project_id=PROJECT_ID, limit=50)

print(f"Found {len(datasets)} dataset(s) in project {PROJECT_ID}\n")

# Display first 5 datasets
for i, ds in enumerate(datasets[:5], 1):
    print(f"{i}. {ds.get('unique_id', 'N/A')}")
    print(f"   Name: {ds.get('dataset_name', 'Unnamed')}")
    print(f"   Measurement: {ds.get('measurement', 'N/A')}")
    print(f"   Public: {ds.get('public', False)}")
    if ds.get('creation_time'):
        print(f"   Created: {ds['creation_time'][:10]}")
    print()

Found 5 dataset(s) in project crucible-demo

1. 0tcy5mfwcxyxs000fs84n0vacw
   Name: Thermal Conductivity Measurement - Sample A (Tutorial)
   Measurement: thermal_conductivity
   Public: False
   Created: 2026-02-24

2. 0tcy5n3hpnxhn0001scz149deg
   Name: Thermal Conductivity Data with Files (Tutorial)
   Measurement: thermal_conductivity
   Public: False
   Created: 2026-02-24

3. 0tcy5q5ytsvf1000h9jaz9kfc8
   Name: Processed Thermal Conductivity Data (Tutorial)
   Measurement: thermal_conductivity_analysis
   Public: False
   Created: 2026-02-24

4. 0tcy5tt115xs7000n802q2fxsr
   Name: Thermal Conductivity Measurement - Sample A (Tutorial)
   Measurement: thermal_conductivity
   Public: False
   Created: 2026-02-24

5. 0tcy5tvd65rjz00039fj0k8nyr
   Name: Thermal Conductivity Data with Files (Tutorial)
   Measurement: thermal_conductivity
   Public: False
   Created: 2026-02-24



<a id='get-dataset'></a>
## 5. Getting a Dataset with Metadata

Retrieve detailed information about a specific dataset, including scientific metadata:

In [7]:
# Get dataset with metadata
dataset_details = client.datasets.get(
    dsid=dataset_id,
    include_metadata=True
)

print(f"Dataset: {dataset_details['unique_id']}")
print(f"Name: {dataset_details.get('dataset_name', 'N/A')}")
print(f"Measurement: {dataset_details.get('measurement', 'N/A')}")
print(f"Public: {dataset_details.get('public', False)}")
print(f"Project: {dataset_details.get('project_id', 'N/A')}")

print(f"\nScientific Metadata:")
if 'scientific_metadata' in dataset_details and dataset_details['scientific_metadata']:
    for key, value in dataset_details['scientific_metadata'].items():
        print(f"  {key}: {value}")
else:
    print("  No metadata available")

Dataset: 0tcy5tt115xs7000n802q2fxsr
Name: Thermal Conductivity Measurement - Sample A (Tutorial)
Measurement: thermal_conductivity
Public: False
Project: crucible-demo

Scientific Metadata:
  id: 107978
  dataset_unique_id: 0tcy5tt115xs7000n802q2fxsr
  scientific_metadata: {'temperature_range': '273-363 K', 'measurement_method': '3-omega method', 'equipment': 'Lakeshore 336 + SR830 lock-in', 'sample_type': 'silicon wafer'}


You can also get keywords and other dataset properties:

In [8]:
# Get keywords
keywords = client.datasets.get_keywords(dataset_id)
if keywords:
    keyword_list = [kw.get('keyword', '') for kw in keywords]
    print(f"Keywords: {', '.join(keyword_list)}")
else:
    print("Keywords: None")

# Get thumbnails
thumbnails = client.datasets.get_thumbnails(dataset_id)
print(f"Number of thumbnails: {len(thumbnails)}")

Keywords: thermal, silicon, conductivity, tutorial
Number of thumbnails: 0


<a id='update-metadata'></a>
## 6. Updating Dataset Metadata

You can update scientific metadata for an existing dataset:

In [ ]:
# Update scientific metadata for the dataset
updated_metadata = {
    "temperature_range": "273-363 K",
    "measurement_method": "3-omega method",
    "equipment": "Lakeshore 336 + SR830 lock-in",
    "sample_type": "silicon wafer",
    "calibration_date": "2026-02-20",
    "operator": "Tutorial User"
}

result = client.datasets.update_scientific_metadata(
    dsid=dataset_id,
    metadata=updated_metadata
)

print(f"✓ Scientific metadata updated for dataset {dataset_id}")

# Verify the update
dataset_updated = client.datasets.get(dsid=dataset_id, include_metadata=True)
print(f"\nUpdated Scientific Metadata:")
if 'scientific_metadata' in dataset_updated and dataset_updated['scientific_metadata']:
    sci_meta = dataset_updated['scientific_metadata'].get('scientific_metadata', {})
    for key, value in sci_meta.items():
        print(f"  {key}: {value}")

<a id='download-dataset'></a>
## 7. Downloading Dataset Files

You can download files from datasets that have been ingested:

In [ ]:
# Get download links for dataset files
download_links = client.datasets.get_download_links(dataset_with_files_id)

print(f"Download links for dataset {dataset_with_files_id}:\n")
for file_path, url in download_links.items():
    print(f"  File: {file_path}")
    print(f"  URL: {url}...")
    print()

# Download all files from the dataset to a directory
import tempfile
download_dir = Path(tempfile.mkdtemp())

print(f"Downloading files to: {download_dir}")
downloaded_files = client.datasets.download(
    dsid=dataset_with_files_id,
    output_dir=str(download_dir)
)

print(f"\n✓ Downloaded {len(downloaded_files)} file(s):")
for file_path in downloaded_files:
    print(f"  - {Path(file_path).name}")

Download links for dataset 0tcy7dca5xsxv000f5zq5n047r:

  File: 0tcy7dca5xsxv000f5zq5n047r/0tcy7dca5xsxv000f5zq5n047r_ingest_2026-02-24T155923.110037-0800_19079.json
  URL: https://storage.googleapis.com/mf-storage-prod/0tcy7dca5xsxv000f5zq5n047r/0tcy7d...

  File: 0tcy7dca5xsxv000f5zq5n047r/measurement_notes.txt
  URL: https://storage.googleapis.com/mf-storage-prod/0tcy7dca5xsxv000f5zq5n047r/measur...

  File: 0tcy7dca5xsxv000f5zq5n047r/thermal_conductivity_data.csv
  URL: https://storage.googleapis.com/mf-storage-prod/0tcy7dca5xsxv000f5zq5n047r/therma...


✓ Downloaded 3 file(s):
  - 0tcy7dca5xsxv000f5zq5n047r_ingest_2026-02-24T155923.110037-0800_19079.json
  - measurement_notes.txt
  - thermal_conductivity_data.csv


<a id='link-sample-dataset'></a>
## 8. Linking a Sample to a Dataset

Associate a dataset with a sample to indicate which sample the data comes from:

In [ ]:
# Link sample to dataset
result = client.samples.add_dataset(
    sample_id=sample_id,
    dataset_id=dataset_id
)

print(f"✓ Sample {sample_id} linked to dataset {dataset_id}")

# Verify the link
datasets_for_sample = client.datasets.list(sample_id=sample_id)
print(f"\nDatasets linked to sample {sample_id}: {len(datasets_for_sample)}")
for ds in datasets_for_sample:
    print(f"  - {ds['unique_id']}: {ds.get('dataset_name', 'N/A')}")

<a id='link-datasets'></a>
## 9. Linking Two Datasets (Parent-Child)

Create hierarchical relationships between datasets. For example, link a processed dataset to its raw data:

In [ ]:
# Create a second dataset (processed data)
processed_dataset = Dataset(
    project_id=PROJECT_ID,
    measurement="thermal_conductivity_analysis",
    dataset_name="Processed Thermal Conductivity Data (Tutorial)",
    public=False
)

result_processed = client.datasets.create(
    dataset=processed_dataset,
    scientific_metadata={
        "thermal_conductivity_300K": 148.5,
        "thermal_conductivity_unit": "W/m·K",
        "processing_method": "polynomial curve fitting (order 2)",
        "uncertainty": 1.8,
        "parent_dataset": dataset_with_files_id
    },
    keywords=["processed", "thermal", "conductivity", "analysis", "tutorial"]
)

processed_dataset_id = result_processed['dsid']
print(f"✓ Processed dataset created: {processed_dataset_id}")

# Link datasets: raw data (parent) -> processed data (child)
link_result = client.datasets.link_parent_child(
    parent_dataset_id=dataset_with_files_id,
    child_dataset_id=processed_dataset_id
)

print(f"\n✓ Datasets linked successfully!")
print(f"  Parent (raw data): {dataset_with_files_id}")
print(f"  Child (processed): {processed_dataset_id}")

# List children of parent dataset
children = client.datasets.list_children(dataset_with_files_id)
print(f"\nChild datasets of {dataset_with_files_id}: {len(children)}")
for child in children:
    print(f"  - {child['unique_id']}: {child.get('dataset_name', 'N/A')}")

# List parents of child dataset
parents = client.datasets.list_parents(processed_dataset_id)
print(f"\nParent datasets of {processed_dataset_id}: {len(parents)}")
for parent in parents:
    print(f"  - {parent['unique_id']}: {parent.get('dataset_name', 'N/A')}")

In [ ]:
# Create a subsample
subsample = client.samples.create(
    sample_name="Silicon Wafer A - Region 1 (Tutorial)",
    project_id=PROJECT_ID,
    description="Sub-region of wafer A for localized measurements (tutorial example)"
)

subsample_id = subsample['unique_id']
print(f"✓ Subsample created: {subsample_id}")

# Link samples: parent sample -> subsample
link_result = client.samples.link(
    parent_id=sample_id,
    child_id=subsample_id
)

print(f"\n✓ Samples linked successfully!")
print(f"  Parent: {sample_id}")
print(f"  Child: {subsample_id}")

# List children of parent sample
children = client.samples.list_children(sample_id)
print(f"\nChild samples of {sample_id}: {len(children)}")
for child in children:
    print(f"  - {child['unique_id']}: {child.get('sample_name', 'N/A')}")

# List parents of child sample
parents = client.samples.list_parents(subsample_id)
print(f"\nParent samples of {subsample_id}: {len(parents)}")
for parent in parents:
    print(f"  - {parent['unique_id']}: {parent.get('sample_name', 'N/A')}")

<a id='link-samples'></a>
## 10. Linking Two Samples (Parent-Child)

Create hierarchical relationships between samples. For example, link a subsample to its parent sample:

In [ ]:
# Path to thumbnail image
thumbnail_path = str(DATA_DIR / "thermal_measurement_preview.png")

# Verify file exists
if Path(thumbnail_path).exists():
    print(f"✓ Thumbnail file found: {Path(thumbnail_path).name}")
    
    # Add thumbnail to dataset
    result = client.datasets.add_thumbnail(
        dsid=dataset_id,
        file_path=thumbnail_path,
        thumbnail_name="thermal_measurement_preview"
    )
    
    print(f"\n✓ Thumbnail added to dataset {dataset_id}")
    
    # List all thumbnails for the dataset
    thumbnails = client.datasets.get_thumbnails(dataset_id)
    print(f"\nThumbnails for dataset:")
    for thumb in thumbnails:
        print(f"  - {thumb.get('name', 'unnamed')}")
else:
    print(f"✗ Thumbnail file not found: {thumbnail_path}")

<a id='add-thumbnail'></a>
## 11. Adding a Thumbnail to a Dataset

Upload a thumbnail image to provide a visual preview of your dataset:

In [ ]:
# Display all IDs created
print("Resource IDs created in this tutorial:")
print(f"\nSamples:")
print(f"  sample_id = {sample_id}")
print(f"  subsample_id = {subsample_id}")
print(f"\nDatasets:")
print(f"  dataset_id = {dataset_id}")
print(f"  dataset_with_files_id = {dataset_with_files_id}")
print(f"  processed_dataset_id = {processed_dataset_id}")

print("\n" + "="*60)
print("You can open any of these resources in your browser using:")
print("="*60)
print(f"\n  crucible open {sample_id}")
print(f"  crucible open {dataset_id}")
print(f"  crucible open {dataset_with_files_id}")
print("\nThe 'crucible open' command works with any dataset, sample, or project ID.")

## Summary

This notebook demonstrated the core Crucible operations:

1. ✓ **Configuration** - Set up API credentials with `crucible config init`
2. ✓ **Create Sample** - `client.samples.create()`
3. ✓ **Create Dataset** - `client.datasets.create()` for metadata-only; `BaseParser(...).upload_dataset()` for datasets with files
4. ✓ **List Datasets** - `client.datasets.list(project_id=...)`
5. ✓ **Get Dataset Details** - `client.datasets.get(dsid=..., include_metadata=True)`
6. ✓ **Update Dataset Metadata** - `client.datasets.update_scientific_metadata()`
7. ✓ **Download Dataset Files** - `client.datasets.get_download_links()` and `client.datasets.download()`
8. ✓ **Link Sample to Dataset** - `client.samples.add_dataset(sample_id, dataset_id)`
9. ✓ **Link Datasets** - `client.datasets.link_parent_child()`, `list_children()`, `list_parents()`
10. ✓ **Link Samples** - `client.samples.link()`, `list_children()`, `list_parents()`
11. ✓ **Add Thumbnail** - `client.datasets.add_thumbnail()`

### Resource IDs Created in This Tutorial

The following variables contain IDs of resources created in this notebook:

In [15]:
# Update dataset metadata
# client.datasets.update_scientific_metadata(
#     dataset_id, 
#     {"new_field": "value", "updated_field": "new_value"}
# )

# Add additional keywords
# client.datasets.add_keyword(dataset_id, "new-keyword")

# Upload additional files to existing dataset
# client.datasets.upload_file(dataset_id, "/path/to/file.txt")

# Request SciCat upload
# client.datasets.request_scicat_upload(dataset_id)

# List all samples in a project
samples = client.samples.list(project_id=PROJECT_ID, limit=999999)
print(f"Total samples in project: {len(samples)}")

# List all projects
projects = client.projects.list(limit=9999)
print(f"Total accessible projects: {len(projects)}")

Total samples in project: 2
Total accessible projects: 187


### Additional Resources

- **Documentation**: See `crucible/cli/README.md` for CLI usage
- **API Reference**: Check docstrings in `crucible/resources/` for all available methods
- **Parsers**: See `crucible/parsers/README.md` for how to use and extend parsers (BaseParser, LAMMPSParser, MatEnsembleManagerParser, MatEnsembleRunParser)
- **Data Files**: Example data files used in this tutorial are in `examples/data/`